In [34]:
import numpy as np
from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
import pandas as pd
from typing import Literal, Tuple, Union, List
import os
import glob
from pathlib import Path
import cv2
import tqdm

# Functions

In [3]:
def spatial_lbp_histogram(image: np.ndarray,
                          P: int = 8,
                          R: int = 2,
                          method: Literal['default', 'ror', 'uniform', 'nri_uniform', 'var'] = 'nri_uniform',
                          grid_x: int = 2,
                          grid_y: int = 2):
   
    assert isinstance(image, np.ndarray) and len(image.shape) == 2, "Input must be a 2D grayscale image."

    lbp_desc = local_binary_pattern(image, P, R, method=method)

    if method == 'uniform':
        n_bins = P + 2
    elif method == 'nri_uniform':
        n_bins = P * (P - 1) + 3
    else: # 'default', 'ror', 'var'
        n_bins = 2**P
        
    h, w = lbp_desc.shape
    cell_h, cell_w = h // grid_y, w // grid_x

    full_hist = []
    for y in range(grid_y):
        for x in range(grid_x):
            y_start, y_end = y * cell_h, (y + 1) * cell_h
            x_start, x_end = x * cell_w, (x + 1) * cell_w
            
            cell = lbp_desc[y_start:y_end, x_start:x_end]
            
            hist, _ = np.histogram(cell.ravel(),
                                   bins=n_bins,
                                   range=(0, n_bins),
                                   density=True)
            
            full_hist.append(hist)
    
    return np.concatenate(full_hist)

In [24]:
def extract_all_lbp(dataset_origin, dest_path, grid: Tuple[int, int], P, R, method):
    dest_path.mkdir(exist_ok=True)
    
    all_features = []
    all_labels = []
    all_filenames = []
    
    class_folders = [p for p in dataset_origin.iterdir() if p.is_dir()]
    
    for class_folder in tqdm.tqdm(class_folders, desc="Processing Images"):
        class_name = class_folder.name
    
        
        for image_path in glob.glob(os.path.join(class_folder, "*.jpg")):
            image = cv2.imread(str(image_path))
            image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            features = spatial_lbp_histogram(image, P= P, R= R, method= method, grid_x=grid[0], grid_y=grid[1])
    
            all_features.append(features)
            all_labels.append(class_name)
            all_filenames.append(Path(image_path).name)
    
    if all_features:
        num_features = len(all_features[0])
        
        feature_columns = [f"feature_{i}" for i in range(num_features)]
        
        features_df = pd.DataFrame(all_features, columns=feature_columns)
        
        features_df['label'] = all_labels
        features_df['filename'] = all_filenames
        
        label_col = features_df.pop('label')
        features_df.insert(0, 'label', label_col)

        features_df.to_csv(dest_path/f"lbp_{P}_{R}_{method}_{grid[0]}x{grid[1]}.csv")

In [29]:
def glcm(img : np.ndarray, 
         distances : Union[List[int],np.ndarray] =[1,3,5], 
         angles : Union[List[float],np.ndarray] = np.deg2rad([0,90,180,270])):
    
    assert isinstance(img, np.ndarray) and len(img.shape) == 2
    hists = graycomatrix(img, distances=distances, angles=angles, normed=True, symmetric=True)
    prop_names = ["contrast", "dissimilarity", "homogeneity", "ASM", "energy", "correlation"]
    props = np.array([ graycoprops(hists, prop).flatten() for prop in prop_names]).flatten()
    
    return props

In [30]:
def spatial_glcm(image: np.ndarray,
                 grid_x: int,
                 grid_y: int,
                 distances: list,
                 angles: list):
    
    h, w = image.shape
    cell_h, cell_w = h // grid_y, w // grid_x
    
    full_features = []
    for y in range(grid_y):
        for x in range(grid_x):
            cell_image = image[y*cell_h:(y+1)*cell_h, x*cell_w:(x+1)*cell_w]     
            cell_features = glcm(cell_image, distances=distances, angles=angles)
            full_features.append(cell_features)
    
    return np.concatenate(full_features)

In [31]:
def extract_all_glcm(dataset_origin: str,
                     dest_path: str,
                     grid: Tuple[int, int],
                     distances: List[int],
                     angles: List[float]):
    
    source_path = Path(dataset_origin)
    dest_path = Path(dest_path)
    dest_path.mkdir(parents=True, exist_ok=True)
    assert source_path.is_dir(), f"Source directory not found: {source_path}"
    
    # --- Unpack grid and print parameters ---
    grid_x, grid_y = grid
    angles_deg_str = np.rad2deg(angles).astype(int)
    print(f"Using GLCM with grid={grid}, distances={distances}, angles={angles_deg_str}°")

    all_features = []
    all_labels = []

    class_folders = [p for p in source_path.iterdir() if p.is_dir()]

    for class_folder in tqdm.tqdm(class_folders, desc="Extracting Spatial GLCM"):
        class_name = class_folder.name
        
        for image_path in class_folder.glob("*.jpg"):
            image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
            
            if image is not None:
                # --- Call the NEW spatial function ---
                features = spatial_glcm(image, grid_x=grid_x, grid_y=grid_y,
                                        distances=distances, angles=angles)
                all_features.append(features)
                all_labels.append(class_name)

    if not all_features:
        print("No features extracted.")
        return

    # --- Create Descriptive Column Names for the Spatial Grid ---
    prop_names = ["contrast", "dissimilarity", "homogeneity", "ASM", "energy", "correlation"]
    angles_deg = np.rad2deg(angles).astype(int)
    num_cells = grid_x * grid_y
    
    feature_columns = [f"cell{cell_idx}_{prop}_d{d}_a{a}"
                       for cell_idx in range(num_cells)
                       for prop in prop_names
                       for d in distances
                       for a in angles_deg]
    
    features_df = pd.DataFrame(all_features, columns=feature_columns)
    features_df['label'] = all_labels
    features_df.insert(0, 'label', features_df.pop('label'))

    # --- Create a descriptive filename including the grid ---
    dist_str = '-'.join(map(str, distances))
    angle_str = '-'.join(map(str, angles_deg))
    output_filename = f"features_glcm_grid{grid_x}x{grid_y}_d{dist_str}_a{angle_str}.csv"
    
    features_df.to_csv(dest_path / output_filename, index=False)

# Code

## Turn images into Gray

In [21]:
dataset_origin = Path("images")
dest_path = Path('gray_images')
dest_path.mkdir(exist_ok=True)

class_folders = [p for p in dataset_origin.iterdir() if p.is_dir()]

for class_folder in tqdm.tqdm(class_folders, desc="Processing Images"):
    class_name = class_folder.name

    dest_class_path = dest_path / class_name
    dest_class_path.mkdir(exist_ok=True)
    
    for image_path in glob.glob(os.path.join(class_folder, "*.jpg")):
        image = cv2.imread(str(image_path))
        gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        image_path = Path(image_path)
        output_path = dest_class_path / image_path.name
        cv2.imwrite(str(output_path), gray_image)

Processing Images: 100%|██████████| 10/10 [00:04<00:00,  2.12it/s]


## Extract LBP and GLCM features

In [21]:
params_tier2 = [
    # --- P=8 (Standard number of points) ---
    # Varying grid and radius
    {'grid': (2, 2), 'P': 8, 'R': 1, 'method': 'nri_uniform'},
    {'grid': (2, 2), 'P': 8, 'R': 2, 'method': 'nri_uniform'},
    {'grid': (4, 4), 'P': 8, 'R': 1, 'method': 'nri_uniform'},
    {'grid': (4, 4), 'P': 8, 'R': 2, 'method': 'nri_uniform'},
    {'grid': (4, 4), 'P': 8, 'R': 3, 'method': 'nri_uniform'},

    # Comparing uniform vs. nri_uniform for a good configuration
    {'grid': (4, 4), 'P': 8, 'R': 2, 'method': 'uniform'},
    
    # Baseline: No spatial grid (global histogram)
    {'grid': (1, 1), 'P': 8, 'R': 2, 'method': 'nri_uniform'},

    # --- P=16 (More angular detail) ---
    # With a larger number of points, we typically use a larger radius
    {'grid': (3, 3), 'P': 16, 'R': 2, 'method': 'uniform'},
    {'grid': (3, 3), 'P': 16, 'R': 3, 'method': 'uniform'},
    {'grid': (4, 4), 'P': 16, 'R': 2, 'method': 'nri_uniform'},
]

In [23]:
for i, params in enumerate(params_tier2):
    print(f"\n--- Running Experiment {i+1}/{len(params_tier2)} ---")
    print(f"Parameters: {params}")

    extract_all_lbp(
        dataset_origin=Path("gray_images"),
        dest_path=Path("lbp_features"),
        grid=params['grid'],
        P=params['P'],
        R=params['R'],
        method=params['method']
    )


--- Running Experiment 1/10 ---
Parameters: {'grid': (2, 2), 'P': 8, 'R': 1, 'method': 'nri_uniform'}


Processing Images: 100%|██████████| 10/10 [01:12<00:00,  7.29s/it]



--- Running Experiment 2/10 ---
Parameters: {'grid': (2, 2), 'P': 8, 'R': 2, 'method': 'nri_uniform'}


Processing Images: 100%|██████████| 10/10 [01:01<00:00,  6.14s/it]



--- Running Experiment 3/10 ---
Parameters: {'grid': (4, 4), 'P': 8, 'R': 1, 'method': 'nri_uniform'}


...

Processing Images: 100%|██████████| 10/10 [01:24<00:00,  8.41s/it]



--- Running Experiment 9/10 ---
Parameters: {'grid': (3, 3), 'P': 16, 'R': 3, 'method': 'uniform'}


Processing Images: 100%|██████████| 10/10 [01:23<00:00,  8.31s/it]



--- Running Experiment 10/10 ---
Parameters: {'grid': (4, 4), 'P': 16, 'R': 2, 'method': 'nri_uniform'}


Processing Images: 100%|██████████| 10/10 [01:29<00:00,  8.98s/it]


In [37]:
params_glcm_tier2 = [
    {'grid': (1, 1), 'distances': [1, 5], 'angles': np.deg2rad([0, 45, 90, 135])},
    {'grid': (4, 4), 'distances': [1, 3], 'angles': np.deg2rad([0, 90])},
    {'grid': (4, 4), 'distances': [1, 5], 'angles': np.deg2rad([0, 45, 90, 135])},
]

In [38]:
for i, params in enumerate(params_glcm_tier2):
    print(f"\n--- Running Experiment {i+1}/{len(params_glcm_tier2)} ---")
    
    extract_all_glcm(
        dataset_origin="gray_images",
        dest_path="glcm_features",
        grid=params['grid'],
        distances=params['distances'],
        angles=params['angles']
    )


--- Running Experiment 1/3 ---
Using GLCM with grid=(1, 1), distances=[1, 5], angles=[  0  45  90 135]°


Extracting Spatial GLCM: 100%|██████████| 10/10 [01:18<00:00,  7.87s/it]



--- Running Experiment 2/3 ---
Using GLCM with grid=(4, 4), distances=[1, 3], angles=[ 0 90]°


Extracting Spatial GLCM: 100%|██████████| 10/10 [10:54<00:00, 65.42s/it]



--- Running Experiment 3/3 ---
Using GLCM with grid=(4, 4), distances=[1, 5], angles=[  0  45  90 135]°


Extracting Spatial GLCM: 100%|██████████| 10/10 [17:18<00:00, 103.88s/it]
